In [ ]:
import yfinance as yf
import pandas as pd
from tqdm import tqdm
from uuid import uuid4
from datetime import datetime, timezone
import json


In [19]:
ticker_symbols = {
    "NVDA": "NVIDIA Corporation",
    "GOOGL": "Alphabet Inc.",
    "AAPL": "Apple Inc.",
    "MSFT": "Microsoft Corporation",
    "AMZN": "Amazon.com, Inc.",
    "TSM": "Taiwan Semiconductor Manufacturing",
    "AVGO": "Broadcom Inc.",
    "2222.SR": "Saudi Arabian Oil Co.",
    "TSLA": "Tesla, Inc.",
    "META": "Meta Platforms, Inc.",
    "MU": "Micron Technology, Inc.",
    "LLY": "Eli Lilly and Company",
    "WMT": "Walmart Inc.",
    "AMD": "Advanced Micro Devices, Inc.",
    "JPM": "JPMorgan Chase & Co.",
    "ASML": "ASML Holding N.V.",
    "V": "Visa Inc.",
    "INTC": "Intel Corporation",
    "XOM": "Exxon Mobil Corporation",
    "JNJ": "Johnson & Johnson",
    "ORCL": "Oracle Corporation",
    "LRCX": "Lam Research Corporation",
    "CSCO": "Cisco Systems Inc.",
    "AMAT": "Applied Materials Inc.",
    "CAT": "Caterpillar Inc.",
    "MA": "Mastercard Incorporated",
    "COST": "Costco Wholesale Corporation",
    "UNH": "UnitedHealth Group Incorporated",
    "ARM": "Arm Holdings plc"
}

In [ ]:
all_hist_data = list()
price_errors = list()
pipeline_run_id = str(uuid4())

for ticker_symbol in tqdm(ticker_symbols):
    fetched_at = datetime.now(timezone.utc)

    try:
        ticker = yf.Ticker(ticker_symbol)
        historical_data = ticker.history(period="10y", interval='1d', auto_adjust=False) 
        
        historical_data["ticker_requested"] = ticker_symbol
        historical_data["source"] = "yfinance"
        historical_data["fetched_at"] = fetched_at
        historical_data["pipeline_run_id"] = pipeline_run_id
        historical_data["period_requested"] = "10y"
        historical_data["interval_requested"] = "1d"
        historical_data["fetch_status"] = "success"
        
        all_hist_data.append(historical_data)

    except Exception as error:
        price_errors.append({
        "ticker_requested": ticker_symbol,
        "source": "yfinance",
        "fetched_at": fetched_at,
        "pipeline_run_id": pipeline_run_id,
        "period_requested": "10y",
        "interval_requested": "1d",
        "fetch_status": "failed",
        "error_message": str(error),
})
    

bronze_market_prices = pd.concat(all_hist_data).reset_index()

100%|██████████| 29/29 [00:03<00:00,  7.45it/s]


In [38]:
bronze_market_prices

,Date,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,ticker_requested,source,fetched_at,pipeline_run_id,period_requested,interval_requested,fetch_status
0,2016-06-20 00:00:00-04:00,1.186750,1.204250,1.186000,1.189000,1.165939,293768000,0.0,0.0,NVDA,yfinance,2026-06-19 15:11:16.178779+00:00,b4de7e34-7492-4216-ba1c-fceb1c695320,10y,1d,success
1,2016-06-21 00:00:00-04:00,1.194000,1.197250,1.180000,1.181750,1.158829,212508000,0.0,0.0,NVDA,yfinance,2026-06-19 15:11:16.178779+00:00,b4de7e34-7492-4216-ba1c-fceb1c695320,10y,1d,success
2,2016-06-22 00:00:00-04:00,1.184250,1.192000,1.178750,1.180750,1.157849,202620000,0.0,0.0,NVDA,yfinance,2026-06-19 15:11:16.178779+00:00,b4de7e34-7492-4216-ba1c-fceb1c695320,10y,1d,success
3,2016-06-23 00:00:00-04:00,1.192000,1.213500,1.191250,1.212250,1.188738,297880000,0.0,0.0,NVDA,yfinance,2026-06-19 15:11:16.178779+00:00,b4de7e34-7492-4216-ba1c-fceb1c695320,10y,1d,success
4,2016-06-24 00:00:00-04:00,1.162500,1.184000,1.132500,1.143250,1.121077,1017684000,0.0,0.0,NVDA,yfinance,2026-06-19 15:11:16.178779+00:00,b4de7e34-7492-4216-ba1c-fceb1c695320,10y,1d,success
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70195,2026-06-12 00:00:00-04:00,353.119995,385.799988,350.040009,380.809998,380.809998,16494100,0.0,0.0,ARM,yfinance,2026-06-19 15:11:20.017198+00:00,b4de7e34-7492-4216-ba1c-fceb1c695320,10y,1d,success
70196,2026-06-15 00:00:00-04:00,390.500000,416.174011,369.250000,412.549988,412.549988,14035500,0.0,0.0,ARM,yfinance,2026-06-19 15:11:20.017198+00:00,b4de7e34-7492-4216-ba1c-fceb1c695320,10y,1d,success
70197,2026-06-16 00:00:00-04:00,401.000000,428.600006,395.880005,396.339996,396.339996,12402000,0.0,0.0,ARM,yfinance,2026-06-19 15:11:20.017198+00:00,b4de7e34-7492-4216-ba1c-fceb1c695320,10y,1d,success
70198,2026-06-17 00:00:00-04:00,411.980011,444.799988,400.140015,418.880005,418.880005,11692500,0.0,0.0,ARM,yfinance,2026-06-19 15:11:20.017198+00:00,b4de7e34-7492-4216-ba1c-fceb1c695320,10y,1d,success


In [ ]:
all_company_data = list()
company_errors = list()
for ticker_symbol in tqdm(ticker_symbols):
    fetched_at = datetime.now(timezone.utc)
    try: 
        ticker = yf.Ticker(ticker_symbol)
        info_dict = ticker.get_info()

        historical_data["fetch_status"] = "success"
        company_data = {
        "ticker_requested": ticker_symbol,
        "source": "yfinance",
        "fetched_at": fetched_at,
        "ticker": ticker_symbol,
        "pipeline_run_id": pipeline_run_id,
        "fetch_status": "success",
        "company_name": info_dict.get("longName"),
        "city": info_dict.get("city"),
        "country": info_dict.get("country"),
        "country": info_dict.get("region"),
        "website": info_dict.get("website"),
        "industry": info_dict.get("industry"),
        "sector": info_dict.get("sector"),
        "currency": info_dict.get("currency"),
        "market": info_dict.get("market"),
        "full_time_employee_count": info_dict.get("fullTimeEmployees"),
        "raw_payload": json.dumps(
                info_dict,
                default=str,
            ),
    }
   
    except Exception as error:
        company_errors.append({
            "ticker_requested": ticker_symbol,
            "source": "yfinance",
            "fetched_at": fetched_at,
            "pipeline_run_id": pipeline_run_id,
            "fetch_status": "failed",
            "error_message": str(error),
        })
    
    
    all_company_data.append(company_data)

bronze_company_metadata = pd.DataFrame(all_company_data)
bronze_company_errors = pd.DataFrame(company_errors)

100%|██████████| 29/29 [00:05<00:00,  5.08it/s]


In [39]:
info_dict

{'address1': '110 Fulbourn Road',
 'city': 'Cambridge',
 'zip': 'CB1 9NJ',
 'country': 'United Kingdom',
 'phone': '44 1223 400 400',
 'website': 'https://www.arm.com',
 'industry': 'Semiconductors',
 'industryKey': 'semiconductors',
 'industryDisp': 'Semiconductors',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Arm Holdings plc researches, develops, licenses, and markets central processing unit (CPU) intellectual property (IP), graphics processing unit IP, systems IP, compute subsystems (CSS), and associated software, tools and related services. The company provides a product portfolio, including CPU IP, GPU and neural processing unit (NPU) accelerators, system IP such as interconnects, compute platform products including pre-integrated CSSs, and development tools and software. The company serves semiconductor companies, original equipment manufacturers (OEMs), cloud service providers (CSPs), and organizations developing ch

In [14]:
info_dict

{'address1': '110 Fulbourn Road',
 'city': 'Cambridge',
 'zip': 'CB1 9NJ',
 'country': 'United Kingdom',
 'phone': '44 1223 400 400',
 'website': 'https://www.arm.com',
 'industry': 'Semiconductors',
 'industryKey': 'semiconductors',
 'industryDisp': 'Semiconductors',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Arm Holdings plc researches, develops, licenses, and markets central processing unit (CPU) intellectual property (IP), graphics processing unit IP, systems IP, compute subsystems (CSS), and associated software, tools and related services. The company provides a product portfolio, including CPU IP, GPU and neural processing unit (NPU) accelerators, system IP such as interconnects, compute platform products including pre-integrated CSSs, and development tools and software. The company serves semiconductor companies, original equipment manufacturers (OEMs), cloud service providers (CSPs), and organizations developing ch

{'address1': 'One Apple Park Way',
 'country': 'United States',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'sector': 'Technology'}